In [1]:
!pip install google-adk google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.1
    Uninstalling cachetools-6.2.1:
      Successfully uninstalled cachetools-6.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Add 'GOOGLE_API_KEY' to Kaggle secrets. Details: {e}")

✅ Setup and authentication complete.


In [3]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
import json
import time
from pathlib import Path
from google.genai import types

In [4]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

In [5]:
MODEL_KW = dict(model="gemini-2.5-flash-lite", retry_options=retry_config)

# 1) Requirements extractor
req_agent = Agent(
    name="ReqExtractor",
    model=Gemini(**MODEL_KW),
    instruction="""
You are a system-design assistant. Given the user prompt
1) Produce Functional Requirements (3-8 bullets).
2) Produce Non-Functional Requirements (latency, availability, scale targets, durability).
3) List 3 assumptions with why they were chosen.
Return valid JSON with keys: functional (list), non_functional (list), assumptions (list of strings).
""",
    output_key="requirements",
)

In [6]:
# 2) Components proposer (can optionally use google_search for grounding)
component_agent = Agent(
    name="ComponentAgent",
    model=Gemini(**MODEL_KW),
    instruction="""
Read requirements: {requirements}
Propose a list of system components. For each component provide:
- name
- purpose (one-sentence)
- suggested tech options (2 choices)
- risk (one line)
Return JSON array.
""",
    output_key="components",
    # tools=[google_search],  # uncomment if you want live lookup
)

# 3) Flow & APIs
flow_agent = Agent(
    name="FlowAgent",
    model=Gemini(**MODEL_KW),
    instruction="""
Given components: {components} and requirements: {requirements}
1) Create 3 main user flows (e.g., Create, Read, Background Processing) step-by-step.
2) Provide API endpoints (method, path, short description, example request/response).
Return JSON { "flows": [...], "apis": [...] }.
""",
    output_key="flows_apis",
)

# 4) Sizing & rough cost
sizing_agent = Agent(
    name="SizingAgent",
    model=Gemini(**MODEL_KW),
    instruction="""
Using requirements: {requirements} and flows: {flows_apis}
If no explicit traffic is given, assume target = 10M monthly active users (~4k-6k QPS peak). 
Provide:
- qps estimates
- storage estimates (per day and for 1 year)
- cache recommendations (size & hit ratio)
- an estimated monthly infra cost range (low, medium, high) and assumptions
Return JSON.
""",
    output_key="sizing",
)

# 5) Trade-offs
tradeoff_agent = Agent(
    name="TradeoffAgent",
    model=Gemini(**MODEL_KW),
    instruction="""
List 5 trade-offs for this design. For each: description, pros, cons, mitigations.
Return JSON array.
""",
    output_key="tradeoffs",
)

# 6) Diagram (Mermaid)
diagram_agent = Agent(
    name="DiagramAgent",
    model=Gemini(**MODEL_KW),
    instruction="""
Using components and flows, output a Mermaid (flowchart) architecture diagram text and a short legend mapping component names to node labels.
Return JSON: { "mermaid": "```mermaid\nflowchart LR\n...\n```", "legend": {...} }.
""",
    output_key="diagram",
)

# 7) Explainer
explainer_agent = Agent(
    name="ExplainerAgent",
    model=Gemini(**MODEL_KW),
    instruction="""
Write a concise 200-350 word explainer targeted at an interviewer:
Include the key design decisions, why chosen, and how the design meets non-functional requirements.
Return plain text.
""",
    output_key="explainer",
)

# 8) Packager
packager_agent = Agent(
    name="PackagerAgent",
    model=Gemini(**MODEL_KW),
    instruction="""
Assemble a final JSON object with keys:
id, prompt, requirements, components, flows_apis, sizing, tradeoffs, diagram, explainer, timestamp
Return valid JSON.
""",
    output_key="final_package",
)

In [7]:
# 9) Root sequential pipeline with parallel component/flow
root_agent = SequentialAgent(
    name="SystemDesignPipeline",
    sub_agents=[
        req_agent,
        component_agent, 
        flow_agent,
        sizing_agent,
        tradeoff_agent,
        diagram_agent,
        explainer_agent,
        packager_agent
    ],
)

In [8]:
runner = InMemoryRunner(agent=root_agent)
print("✅ SystemDesignPipeline agent tree created.")
response = await runner.run_debug(
    "Design a scalable system to handle real-time fraud detection for a global e-commerce platform with 50 million daily transactions. The system should support streaming ingestion, low-latency inference, model retraining, alerting, and dashboards. Provide the architecture, components, storage layers, APIs, ML model choices, and challenges."
)

✅ SystemDesignPipeline agent tree created.

 ### Created new session: debug_session_id

User > Design a scalable system to handle real-time fraud detection for a global e-commerce platform with 50 million daily transactions. The system should support streaming ingestion, low-latency inference, model retraining, alerting, and dashboards. Provide the architecture, components, storage layers, APIs, ML model choices, and challenges.
ReqExtractor > ```json
{
  "functional": [
    "Ingest transaction data in real-time from various sources (e.g., web, mobile, API gateways).",
    "Perform real-time fraud scoring for each transaction with sub-second latency.",
    "Trigger alerts to relevant stakeholders (e.g., fraud analysts, customer support) when suspicious transactions are detected.",
    "Provide a dashboard for monitoring fraud trends, alert statuses, and system performance.",
    "Support batch and/or stream-based retraining of fraud detection models.",
    "Store historical transaction